# 02 ACDC Traversal Ordering

Replicate the ACDC traversal-ordering comparison. Receiver nodes follow reverse topology; only parallel stages inside a layer are randomized for the random-order condition.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from circuit_discovery.run import (
    get_compute_device,
    load_configs,
    load_model,
    load_task_dataset_from_config,
    load_circuit_map,
    evaluation_rows,
    pairwise_iou_rows,
    train_loader_from_config,
)

configs = load_configs()
print("project root:", PROJECT_ROOT)
print("device:", get_compute_device())

params = configs["notebooks"]["02_acdc_traversal_ordering"]["hyperparams"]
artifacts = configs["artifacts"]["acdc"]["circuits"]
params


In [ ]:
# Optional regeneration. Expensive; leave disabled when browsing saved artifacts.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    from circuit_discovery.algorithms.acdc import ACDC, ACDCConfig
    from circuit_discovery.algorithms.edge_pruning import EdgePruningTeacherCache
    from circuit_discovery.metrics import discogp_fidelity_loss, edge_pruning_full_vocab_kl_loss

    output_dir = PROJECT_ROOT / configs["artifacts"]["acdc"]["root"]
    output_dir.mkdir(parents=True, exist_ok=True)
    data = load_task_dataset_from_config(params)
    model = load_model(params["model_name"])

    if params["objective"] == "kl_full_vocab":
        teacher_cache = EdgePruningTeacherCache(
            model=model,
            dtype=torch.float16,
            storage_device="cpu",
        )

        def loss_fn(batch, logits):
            teacher_logits = teacher_cache.logits_for_batch(batch)
            return edge_pruning_full_vocab_kl_loss(batch, logits, teacher_logits)

    elif params["objective"] == "two_label":
        loss_fn = discogp_fidelity_loss
    else:
        raise ValueError(f"unknown ACDC objective: {params['objective']}")

    for label, ordering in {
        "fixed_order": "fixed",
        "random_per_layer_order_seed_42": "random_per_layer",
    }.items():
        runner = ACDC(
            model=model,
            config=ACDCConfig(
                model_name=params["model_name"],
                thresholds=(params["threshold"],),
                max_batches=params["max_batches"],
                optimized_for_acdc=params["optimized_for_acdc"],
                edge_ordering=ordering,
                seed=params["seed"],
                tqdm_disabled=False,
            ),
            device=get_compute_device(),
        )
        result = runner.fit(
            train_loader_from_config(data.train.dataset, params),
            loss_fn=loss_fn,
        )
        circuit = result.circuit_for_threshold(
            params["threshold"],
            model=model,
            finalize=True,
        )
        torch.save(
            {"circuit": circuit, "algorithm": "acdc", "edge_ordering": ordering},
            output_dir / f"{label}.pt",
        )


In [ ]:
model = load_model(params["model_name"])
data = load_task_dataset_from_config(params)
circuits = load_circuit_map(artifacts)
print("loaded circuits:", list(circuits))
display(evaluation_rows(model, data.test, circuits))


In [ ]:
display(pairwise_iou_rows(circuits))
